# Analisis Kegagalan DDPG — Versi Asli (Tabel VI.1)

Notebook ini membuktikan gejala kegagalan DDPG (dibanding PPO) LANGSUNG dari data pelatihan/evaluasi yang menghasilkan **Tabel VI.1** (skenario kebijakan penuh, backbone `master_pure`/`master_pure_ppo`, 2-aliran/`raw`, horizon 90 hari, `wait-fail-threshold=120`).

Berkas sumber (dikonfirmasi cocok PERSIS dengan angka Tabel VI.1):
- **PPO**: `uji_master_pure_ppo_dgr_90d_cwtfail120pen-2_metrik_90d.json` + `master_pure_ppo_dgr_90d_cwtfail120pen-2_training_results.json`
- **DDPG**: `uji_master_pure_dgr_90d_cwtfail120pen-2_upc62_metrik_90d.json` + `master_pure_dgr_90d_cwtfail120pen-2_upc62_training_results.json`

Kondisi agregat yang dipakai: `signed|dinamis` (aturan trust asimetris, trust berevolusi sungguhan).

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt

OUT = "outputs"

def muat(f):
    return json.load(open(f"{OUT}/{f}", encoding="utf-8"))

def _gini(a):
    a = np.clip(np.asarray(a, dtype=float), 0, None)
    if a.sum() == 0:
        return 0.0
    a = np.sort(a)
    n = a.shape[0]
    idx = np.arange(1, n + 1)
    return float(np.sum((2 * idx - n - 1) * a) / (n * np.sum(a)))

PPO_METRIK = muat("uji_master_pure_ppo_dgr_90d_cwtfail120pen-2_metrik_90d.json")
DDPG_METRIK = muat("uji_master_pure_dgr_90d_cwtfail120pen-2_upc62_metrik_90d.json")
PPO_TRAIN = muat("master_pure_ppo_dgr_90d_cwtfail120pen-2_training_results.json")
DDPG_TRAIN = muat("master_pure_dgr_90d_cwtfail120pen-2_upc62_training_results.json")

print("Berkas berhasil dimuat.")
print("  PPO  n_checkpoint:", PPO_METRIK.get("n_checkpoint"), " | n_seed training:", len(PPO_TRAIN))
print("  DDPG n_checkpoint:", DDPG_METRIK.get("n_checkpoint"), " | n_seed training:", len(DDPG_TRAIN))

## Lapis 0 — Verifikasi: reproduksi Tabel VI.1

Sebelum apa pun, pastikan berkas yang dimuat memang menghasilkan angka yang SAMA PERSIS dengan tabel yang dilaporkan — supaya seluruh analisis di bawah punya provenance yang jelas.

In [ ]:
def ambil_kondisi(d, kondisi="signed|dinamis"):
    a = d["agregat"]
    k = [kk for kk in a if kondisi in kk][0]
    return a[k]

def gini_util(d, kondisi="signed|dinamis"):
    ps = d["per_seed"]
    k = [kk for kk in ps if kondisi in kk][0]
    runs = ps[k]
    gu = np.array([_gini([v["util_mean"] for v in r["_stasiun"].values()]) for r in runs])
    if "ckpt_per_seed" in d and d.get("n_checkpoint"):
        grup = {c: [] for c in range(d["n_checkpoint"])}
        for sd_str, c in d["ckpt_per_seed"].items():
            grup[c].append(gu[int(sd_str)])
        gu = np.array([np.mean(v) for v in grup.values()])
    return float(gu.mean()), float(gu.std())

ppo_row, ddpg_row = ambil_kondisi(PPO_METRIK), ambil_kondisi(DDPG_METRIK)
ppo_gu, ppo_gu_sd = gini_util(PPO_METRIK)
ddpg_gu, ddpg_gu_sd = gini_util(DDPG_METRIK)

print("                 PPO                  DDPG")
print(f"Gini utilisasi   {ppo_gu:.4f} ± {ppo_gu_sd:.4f}      {ddpg_gu:.4f} ± {ddpg_gu_sd:.4f}")
print(f"wait (menit)     {ppo_row['wait']:.1f} ± {ppo_row['wait_sd']:.1f}          {ddpg_row['wait']:.1f} ± {ddpg_row['wait_sd']:.1f}")
print(f"acc              {ppo_row['acc']:.4f} ± {ppo_row['acc_sd']:.4f}      {ddpg_row['acc']:.4f} ± {ddpg_row['acc_sd']:.4f}")
print()
print("Bandingkan dgn Tabel VI.1: PPO 0,0583±0,0365 / 76,3±9,6 / 0,7290±0,0023")
print("                           DDPG 0,1922±0,2105 / 479,3±553,4 / 0,7874±0,1420")

## Lapis 2 — Gejala VARIANSI antar-seed

Bukan cuma rata-rata yang kalah — seberapa KONSISTEN hasil DDPG antar-seed dibanding PPO?

In [ ]:
print(f"Rasio SD/mean wait  -- PPO: {ppo_row['wait_sd']/ppo_row['wait']:.2f}   DDPG: {ddpg_row['wait_sd']/ddpg_row['wait']:.2f}")
print(f"Rasio SD/mean gini_UTIL -- PPO: {ppo_gu_sd/ppo_gu:.2f}   DDPG: {ddpg_gu_sd/ddpg_gu:.2f}")
print()
print("DDPG: SD wait (553,4) LEBIH BESAR dari rata-ratanya sendiri (479,3) -- koefisien variasi >1,")
print("artinya sebaran hasil antar-seed sangat lebar/tak konsisten -- bukan sekadar 'lebih lambat',")
print("tapi 'tidak bisa diprediksi seed mana yang akan berhasil'.")

## Lapis 3a — Gejala DI DALAM proses latih: lonjakan gradien

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for run in PPO_TRAIN:
    gn = [it["grad_norm"] for it in run["history"]]
    axes[0].plot(gn, label=f"seed {run['seed']}", alpha=0.8)
axes[0].set_title("PPO -- grad_norm per iterasi")
axes[0].set_xlabel("iterasi"); axes[0].set_ylabel("grad_norm"); axes[0].legend(); axes[0].set_yscale("log")

for run in DDPG_TRAIN:
    gn = [it["critic_grad"] for it in run["history"]]
    axes[1].plot(gn, label=f"seed {run['seed']}", alpha=0.8)
axes[1].set_title("DDPG -- critic_grad per iterasi")
axes[1].set_xlabel("iterasi"); axes[1].set_ylabel("critic_grad"); axes[1].legend(); axes[1].set_yscale("log")

plt.tight_layout(); plt.savefig(f"{OUT}/_notebook_grad_norm_ppo_vs_ddpg.png", dpi=110)
plt.show()

print("grad_norm MAKSIMUM sepanjang training:")
for run in PPO_TRAIN:
    gn = [it["grad_norm"] for it in run["history"]]
    print(f"  PPO  seed {run['seed']}: maks={max(gn):.2f}  median={np.median(gn):.2f}")
for run in DDPG_TRAIN:
    gn = [it["critic_grad"] for it in run["history"]]
    ag = [it["actor_grad"] for it in run["history"]]
    print(f"  DDPG seed {run['seed']}: critic_grad maks={max(gn):.2f} median={np.median(gn):.2f}  |  actor_grad maks={max(ag):.2f} median={np.median(ag):.2f}")

## Lapis 3b — Eksplorasi MELURUH (DDPG) vs KONSTAN (PPO, lewat entropi)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for run in DDPG_TRAIN:
    ns = [it["noise_std"] for it in run["history"]]
    axes[0].plot(ns, label=f"seed {run['seed']}")
axes[0].set_title("DDPG -- noise_std (eksplorasi) per iterasi")
axes[0].set_xlabel("iterasi"); axes[0].set_ylabel("noise_std"); axes[0].legend()

for run in PPO_TRAIN:
    ent = [it["entropy"] for it in run["history"]]
    axes[1].plot(ent, label=f"seed {run['seed']}")
axes[1].set_title("PPO -- entropi kebijakan per iterasi")
axes[1].set_xlabel("iterasi"); axes[1].set_ylabel("entropi"); axes[1].legend()

plt.tight_layout(); plt.savefig(f"{OUT}/_notebook_eksplorasi_ppo_vs_ddpg.png", dpi=110)
plt.show()

print("DDPG noise_std: awal -> akhir")
for run in DDPG_TRAIN:
    ns = [it["noise_std"] for it in run["history"]]
    print(f"  seed {run['seed']}: {ns[0]:.2f} -> {ns[-1]:.2f}")

## Lapis 3c — Bobot `beta` DGR: kemana perhatian gradien dialokasikan

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)

for run in PPO_TRAIN:
    betas = np.array([it["beta"] for it in run["history"]])
    axes[0].plot(betas[:, 0], alpha=0.6, label=f"wait, seed{run['seed']}")
    axes[0].plot(betas[:, 1], alpha=0.6, ls="--", label=f"gini, seed{run['seed']}")
axes[0].set_title("PPO -- beta (wait vs gini)")
axes[0].set_xlabel("iterasi"); axes[0].set_ylabel("beta"); axes[0].legend(fontsize=7)

for run in DDPG_TRAIN:
    betas = np.array([it["beta"] for it in run["history"]])
    axes[1].plot(betas[:, 0], alpha=0.6, label=f"wait, seed{run['seed']}")
    axes[1].plot(betas[:, 1], alpha=0.6, ls="--", label=f"gini, seed{run['seed']}")
axes[1].set_title("DDPG -- beta (wait vs gini)")
axes[1].set_xlabel("iterasi"); axes[1].legend(fontsize=7)

plt.tight_layout(); plt.savefig(f"{OUT}/_notebook_beta_ppo_vs_ddpg.png", dpi=110)
plt.show()

print("beta gini, 10 iterasi terakhir:")
for run in PPO_TRAIN:
    betas = np.array([it["beta"] for it in run["history"]])
    rm = np.array([it["ret_mean"] for it in run["history"]])
    print(f"  PPO  seed {run['seed']}: beta_gini={betas[-10:,1].mean():.3f}  ret_mean_gini={rm[-10:,1].mean():.4f}")
for run in DDPG_TRAIN:
    betas = np.array([it["beta"] for it in run["history"]])
    print(f"  DDPG seed {run['seed']}: beta_gini={betas[-10:,1].mean():.3f}")

## Lapis 4a — `gini_util` PER ITERASI (snapshot noisy pelatihan) -- lihat apakah membaik atau mendatar

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for run in PPO_TRAIN:
    gu = [it["gini_util"] for it in run["history"]]
    ax.plot(gu, alpha=0.7, label=f"PPO seed{run['seed']}")
for run in DDPG_TRAIN:
    gu = [it["gini_util"] for it in run["history"]]
    ax.plot(gu, alpha=0.7, ls="--", label=f"DDPG seed{run['seed']}")
ax.set_title("gini_util per iterasi (snapshot batch, BERISIK)")
ax.set_xlabel("iterasi"); ax.set_ylabel("gini_util"); ax.legend(fontsize=7)
plt.tight_layout(); plt.savefig(f"{OUT}/_notebook_giniutil_iterasi.png", dpi=110)
plt.show()

## Lapis 4b — DGR vs spesialis-spesialisnya sendiri (kompromi wajar atau kolaps?)

In [ ]:
def r_star_terakhir(fname_list):
    out = []
    for f in fname_list:
        d = muat(f)
        for run in d:
            h = run["history"][-1]
            out.append((run["seed"], h.get("gini_util"), run.get("r_star")))
    return out

print("=== PPO ===")
print("spesialis wait :", r_star_terakhir(["master_pure_ppo_specialist0_wait_90d_cwtfail120pen-2_training_results.json"]))
print("spesialis gini :", r_star_terakhir(["master_pure_ppo_specialist1_gini_90d_cwtfail120pen-2_training_results.json"]))
print("DGR            :", [(r["seed"], r["history"][-1]["gini_util"]) for r in PPO_TRAIN])
print(f"DGR wait (evaluasi penuh)  = {ppo_row['wait']:.1f} menit")
print()
print("=== DDPG ===")
print("spesialis wait :", r_star_terakhir(["master_pure_specialist0_wait_90d_cwtfail120pen-2_upc62_training_results.json"]))
print("spesialis gini :", r_star_terakhir(["master_pure_specialist1_gini_90d_cwtfail120pen-2_upc62_training_results.json"]))
print("DGR            :", [(r["seed"], r["history"][-1]["gini_util"]) for r in DDPG_TRAIN])
print(f"DGR wait (evaluasi penuh)  = {ddpg_row['wait']:.1f} menit")

## Lapis 4c — Throughput (`served`) -- tanda kemacetan sistemik

In [ ]:
print(f"served -- PPO : {ppo_row.get('served')}")
print(f"served -- DDPG: {ddpg_row.get('served')}")
print()
print("Jika DDPG served JAUH lebih rendah drpd PPO, ini indikasi kemacetan riil")
print("(bukan cuma pengguna menunggu lebih lama, tapi lebih SEDIKIT yg berhasil dilayani).")

## Ringkasan

Isi tiap sel di atas dengan angka SUNGGUHAN (bukan estimasi) begitu notebook ini dijalankan sekali —
seluruh grafik (`_notebook_*.png`) tersimpan otomatis di `outputs/`, siap dipakai sbg Lampiran/bukti sidang.

**Kerangka lapis bukti** (rujuk pembahasan sesi ini):
1. Metrik akhir (Lapis 0/1) -- gejala PERMUKAAN.
2. Variansi antar-seed (Lapis 2) -- ketidakstabilan.
3. Dinamika pelatihan: grad_norm, eksplorasi meluruh, bobot beta (Lapis 3) -- mekanisme.
4. Struktural: DGR vs spesialis, throughput (Lapis 4) -- konsekuensi sistemik.